# RGB-Agent Offline Evaluation

This notebook runs the local `RGB-Agent` evaluation flow in `offline` mode on Kaggle.

What it does:
- installs ARC wheels from the Kaggle competition input
- installs this project
- configures the analyzer backend and model
- runs offline evaluation against local `environment_files/`
- saves `scorecard.json`, `summary.txt`, `summary.csv`, and `results.json`

Before running:
- make sure this repo is available in the notebook filesystem
- if you use `direct`, set `OPENAI_BASE_URL` to your OpenAI-compatible endpoint
- if you use `opencode`, Docker support is required and usually not suitable for Kaggle


In [ ]:
from pathlib import Path
import os

PROJECT_CANDIDATES = [
    Path('/kaggle/working/RGB-Agent'),
    Path('/kaggle/input/rgb-agent/RGB-Agent'),
    Path('/kaggle/input/RGB-Agent/RGB-Agent'),
    Path.cwd(),
]

PROJECT_ROOT = None
for candidate in PROJECT_CANDIDATES:
    if (candidate / 'pyproject.toml').exists() and (candidate / 'rgb_agent').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the RGB-Agent project root. Upload or mount the repo first.')

WHEEL_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
ENV_DIR = PROJECT_ROOT / 'environment_files'
OUTPUT_ROOT = Path('/kaggle/working/evaluation_results')

print('PROJECT_ROOT =', PROJECT_ROOT)
print('WHEEL_DIR    =', WHEEL_DIR)
print('ENV_DIR      =', ENV_DIR)
print('OUTPUT_ROOT  =', OUTPUT_ROOT)


In [ ]:
import subprocess
import sys

if not WHEEL_DIR.exists():
    raise FileNotFoundError(f'ARC wheel directory not found: {WHEEL_DIR}')

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--no-index',
    f'--find-links={WHEEL_DIR}',
    'arc-agi',
    'python-dotenv',
    'requests',
])

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    str(PROJECT_ROOT),
])

print('Dependencies installed.')


## Configuration

Edit this cell before running evaluation.

Notes:
- `ANALYZER_BACKEND='direct'` is the Kaggle-friendly default.
- `MODEL` must match the model ID exposed by your endpoint.
- `OPENAI_BASE_URL` is only used by the `direct` backend.
- `SUITE='all'` runs the bundled preview environments.


In [ ]:
AGENT_NAME = 'rgb_agent'
SUITE = 'all'
GAME = None
MAX_ACTIONS = 500
ANALYZER_INTERVAL = 10
ANALYZER_RETRIES = 5
DESCRIPTION = 'kaggle-offline-run'

ANALYZER_BACKEND = 'direct'
MODEL = 'openai/Qwen2.5-72B-Instruct'

# For direct backend, point this at your OpenAI-compatible endpoint.
# Example local server inside the same notebook runtime:
# OPENAI_BASE_URL = 'http://127.0.0.1:8000/v1'
OPENAI_BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:8000/v1')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'EMPTY')

os.environ['ENVIRONMENTS_DIR'] = str(ENV_DIR)
os.environ['ANALYZER_BACKEND'] = ANALYZER_BACKEND
os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print('ANALYZER_BACKEND =', ANALYZER_BACKEND)
print('MODEL            =', MODEL)
print('OPENAI_BASE_URL  =', OPENAI_BASE_URL)


In [ ]:
from rgb_agent.evaluation.offline_eval import run_offline_evaluation

result = run_offline_evaluation(
    agent_name=AGENT_NAME,
    game=GAME,
    suite=SUITE,
    max_actions=MAX_ACTIONS,
    analyzer_interval=ANALYZER_INTERVAL,
    analyzer_model=MODEL,
    analyzer_backend=ANALYZER_BACKEND,
    analyzer_retries=ANALYZER_RETRIES,
    description=DESCRIPTION,
    output_root=OUTPUT_ROOT,
    environments_dir=ENV_DIR,
)

result


In [ ]:
from pathlib import Path
import json

run_dir = Path(result['run_dir'])
print('Run dir:', run_dir)
print('\nArtifacts:')
for path in sorted(run_dir.iterdir()):
    print('-', path.name)

if result.get('scorecard_json'):
    scorecard = json.loads(Path(result['scorecard_json']).read_text())
    print('\nOverall score:', scorecard.get('score'))


In [ ]:
summary_path = Path(result['summary_txt'])
print(summary_path.read_text())


In [ ]:
import pandas as pd

csv_path = Path(result['summary_csv'])
df = pd.read_csv(csv_path)
df
